# Task 2 — Notebook 5: Feature Engineering

This notebook converts the three data splits into model-ready features.

The same feature-building function is applied to train, validation, and test. The preprocessing object is **fitted on the training data only** and then used to transform the other two splits.

**Inputs:** `03_train.parquet`, `03_validation.parquet`, `03_test.parquet`

**Outputs:** transformed feature matrices, target arrays, a fitted preprocessor, and a feature list.


## 1. Imports and paths


In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import sys

src_dir = Path.cwd().resolve()
while src_dir != src_dir.parent and not (src_dir / "Task2").is_dir():
    src_dir = src_dir.parent
src_dir = src_dir / "Task2" / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from config import find_project_root

PROJECT_ROOT = find_project_root()
TASK2_ROOT = PROJECT_ROOT / "Task2"
ARTIFACTS_DIR = TASK2_ROOT / "artifacts"
TABLES_DIR = ARTIFACTS_DIR / "tables"
FEATURES_DIR = ARTIFACTS_DIR / "features"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

for folder in [FEATURES_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

split_paths = {
    "train": TABLES_DIR / "03_train.parquet",
    "validation": TABLES_DIR / "03_validation.parquet",
    "test": TABLES_DIR / "03_test.parquet",
}

missing_files = [str(path) for path in split_paths.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing split files: {missing_files}")


## 2. Load the three splits


In [2]:
train = pd.read_parquet(split_paths["train"])
validation = pd.read_parquet(split_paths["validation"])
test = pd.read_parquet(split_paths["test"])

for name, data in {"train": train, "validation": validation, "test": test}.items():
    assert "is_late" in data.columns, f"Target is missing from {name}"
    print(f"{name.title():<12} rows: {len(data):,}")


Train        rows: 67,529
Validation   rows: 14,470
Test         rows: 14,471


## 3. Build prediction-time features

Date columns are converted into useful calendar variables and pre-delivery durations. Additional order-level ratios are created from price, freight, payment, and item counts. Identifiers and post-outcome information are removed. In particular, actual delivery timestamps and review information are excluded because they are not available when the prediction is made.


In [3]:
def make_features(frame):
    features = frame.copy()

    date_columns = [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_estimated_delivery_date",
    ]
    for column in date_columns:
        if column in features.columns:
            features[column] = pd.to_datetime(features[column], errors="coerce")

    purchase_time = features.get("order_purchase_timestamp")
    if purchase_time is not None:
        features["purchase_year"] = purchase_time.dt.year
        features["purchase_month"] = purchase_time.dt.month
        features["purchase_dayofweek"] = purchase_time.dt.dayofweek
        features["purchase_hour"] = purchase_time.dt.hour

    if "order_approved_at" in features.columns and purchase_time is not None:
        features["approval_hours"] = (
            features["order_approved_at"] - purchase_time
        ).dt.total_seconds() / 3600

    if "order_estimated_delivery_date" in features.columns and purchase_time is not None:
        features["estimated_window_days"] = (
            features["order_estimated_delivery_date"] - purchase_time
        ).dt.total_seconds() / 86400

    # Simple order-level ratios that are available before delivery.
    if {"total_freight", "total_price"}.issubset(features.columns):
        features["freight_to_price_ratio"] = (
            features["total_freight"] / features["total_price"].replace(0, np.nan)
        )

    if {"total_price", "num_items"}.issubset(features.columns):
        features["price_per_item"] = (
            features["total_price"] / features["num_items"].replace(0, np.nan)
        )

    if {"total_freight", "num_items"}.issubset(features.columns):
        features["freight_per_item"] = (
            features["total_freight"] / features["num_items"].replace(0, np.nan)
        )

    if {"total_payment_value", "total_price"}.issubset(features.columns):
        features["payment_price_difference"] = (
            features["total_payment_value"] - features["total_price"]
        )

    if purchase_time is not None:
        features["purchase_weekofyear"] = purchase_time.dt.isocalendar().week.astype("float64")
        features["is_weekend"] = (purchase_time.dt.dayofweek >= 5).astype("int8")

    columns_to_remove = [
        # Target and identifiers.
        "is_late", "order_id", "customer_id", "customer_unique_id",
        # Constant or non-predictive field in this dataset.
        "order_status",
        # Information unavailable at prediction time.
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "avg_review_score",
        "review_record_count",
        # Original date objects are replaced by derived features.
        "order_purchase_timestamp",
        "order_approved_at",
        "order_estimated_delivery_date",
    ]

    return features.drop(
        columns=[column for column in columns_to_remove if column in features.columns]
    )

X_train_raw = make_features(train)
X_validation_raw = make_features(validation)
X_test_raw = make_features(test)

y_train = train["is_late"].astype("int8").to_numpy()
y_validation = validation["is_late"].astype("int8").to_numpy()
y_test = test["is_late"].astype("int8").to_numpy()

# The three raw feature tables must have the same columns and order.
assert list(X_train_raw.columns) == list(X_validation_raw.columns)
assert list(X_train_raw.columns) == list(X_test_raw.columns)

numeric_features = X_train_raw.select_dtypes(include=np.number).columns.tolist()
categorical_features = [
    column for column in X_train_raw.columns if column not in numeric_features
]

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")


Numeric features: 25
Categorical features: 2


## 4. Fit preprocessing on train only

The numeric pipeline fills missing values with the training median and standardizes the values. The categorical pipeline fills missing values with the most common training value and applies one-hot encoding. `handle_unknown='ignore'` allows validation or test to contain a category that was not present in training.


In [4]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

# Fit once on train; transform validation and test without refitting.
X_train = preprocessor.fit_transform(X_train_raw)
X_validation = preprocessor.transform(X_validation_raw)
X_test = preprocessor.transform(X_test_raw)

print("Transformed shapes:")
print(f"  train:      {X_train.shape}")
print(f"  validation: {X_validation.shape}")
print(f"  test:       {X_test.shape}")


Transformed shapes:
  train:      (67529, 808)
  validation: (14470, 808)
  test:       (14471, 808)


## 5. Save model-ready artifacts


In [5]:
def save_sparse_matrix(matrix, path):
    matrix = matrix.tocsr() if sparse.issparse(matrix) else sparse.csr_matrix(matrix)
    sparse.save_npz(path, matrix)

save_sparse_matrix(X_train, FEATURES_DIR / "05_X_train.npz")
save_sparse_matrix(X_validation, FEATURES_DIR / "05_X_validation.npz")
save_sparse_matrix(X_test, FEATURES_DIR / "05_X_test.npz")

np.save(FEATURES_DIR / "05_y_train.npy", y_train)
np.save(FEATURES_DIR / "05_y_validation.npy", y_validation)
np.save(FEATURES_DIR / "05_y_test.npy", y_test)

joblib.dump(preprocessor, FEATURES_DIR / "05_transformers.joblib")
# Keep an explicit production-friendly name for the fitted transformer.
joblib.dump(preprocessor, FEATURES_DIR / "05_preprocessor.joblib")

feature_list = list(preprocessor.get_feature_names_out())
(FEATURES_DIR / "05_feature_list.json").write_text(
    json.dumps(feature_list, indent=2),
    encoding="utf-8",
)

feature_metadata = {
    "artifact_version": "task2-v1",
    "target": "is_late",
    "raw_feature_columns": list(X_train_raw.columns),
    "prediction_time_policy": "Only information available before delivery is allowed.",
    "feature_count": len(feature_list),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "excluded_post_outcome_columns": [
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "avg_review_score",
        "review_record_count",
    ],
    "fit_rule": "Preprocessor fitted on train only.",
    "preprocessor_artifact": "05_preprocessor.joblib",
}
(REPORTS_DIR / "05_feature_metadata.json").write_text(
    json.dumps(feature_metadata, indent=2),
    encoding="utf-8",
)

print(f"Saved {len(feature_list)} transformed features.")


Saved 808 transformed features.


## 6. Final checks


In [6]:
assert X_train.shape[0] == len(y_train)
assert X_validation.shape[0] == len(y_validation)
assert X_test.shape[0] == len(y_test)
assert X_train.shape[1] == X_validation.shape[1] == X_test.shape[1]
assert len(feature_list) == X_train.shape[1]
assert (FEATURES_DIR / "05_transformers.joblib").exists()
assert (FEATURES_DIR / "05_preprocessor.joblib").exists()
assert (FEATURES_DIR / "05_feature_list.json").exists()
assert (REPORTS_DIR / "05_feature_metadata.json").exists()
print("ALL NOTEBOOK 5 CHECKS PASSED")


ALL NOTEBOOK 5 CHECKS PASSED
